# APPROCHE MATRICIELLE : MODELE DE TRANSITION MARKOVIEN
## Application au Remboursement Anticipe (RA)

## Introduction theorique

### Qu'est-ce qu'une chaine de Markov ?

Une chaine de Markov est un processus stochastique ou la probabilite de passer d'un etat a un autre ne depend que de l'etat actuel, pas de l'historique.

**Propriete de Markov** :
$$P(X_{t+1} = j | X_t = i, X_{t-1}, ..., X_0) = P(X_{t+1} = j | X_t = i) = p_{ij}$$

### Application au remboursement anticipe

Dans notre contexte :
- **Etats** = niveaux de remboursement anticipe (ER)
- **Temps** = age du pret (en mois)
- **Matrice de transition** = probabilite de passer d'un niveau de RA a un autre selon l'age

### Pourquoi cette approche est adaptee a nos donnees ?

1. **Cross-section** : une observation par contrat suffit
2. **Agregation par age** : on estime les probabilites de transition par tranche d'age
3. **Interpretabilite** : la matrice est directement lisible et explicable
4. **Utilisation bancaire** : methode standard en ALM pour le RA

### Structure de la matrice de transition

$$M(t) = \begin{pmatrix} p_{11}(t) & p_{12}(t) & p_{13}(t) & p_{14}(t) \\ 0 & p_{22}(t) & p_{23}(t) & p_{24}(t) \\ 0 & 0 & p_{33}(t) & p_{34}(t) \\ 0 & 0 & 0 & 1 \end{pmatrix}$$

ou les etats sont :
- **Etat 1** : Pas de RA (ER = 0)
- **Etat 2** : RA faible (0 < ER <= 0.33)
- **Etat 3** : RA moyen (0.33 < ER <= 0.67)
- **Etat 4** : RA fort (ER > 0.67)

### Propriete d'absorption

L'etat 4 (RA fort) est un **etat absorbant** : une fois qu'un contrat a un RA fort, il ne peut pas revenir en arriere.

La matrice est **triangulaire superieure** : on ne peut qu'augmenter son niveau de RA.

### Projection de l'ER

La distribution des etats a l'age t+k est :

$$\pi(t+k) = \pi(t) \times M(t) \times M(t+1) \times ... \times M(t+k-1)$$

L'ER attendu a l'horizon k est :

$$E[ER(t+k)] = \sum_{s=1}^{4} \pi_s(t+k) \times \bar{ER}_s$$

ou $\bar{ER}_s$ est l'ER moyen dans l'etat s.

## Imports et configuration

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.preprocessing import LabelEncoderfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import mean_squared_error, mean_absolute_error, r2_scoreimport warningswarnings.filterwarnings('ignore')plt.style.use('seaborn-v0_8-darkgrid')sns.set_palette("husl")print("Bibliotheques chargees avec succes")

## Etape 1 : Chargement et preparation des donnees

In [ ]:
# Chargementdf = pd.read_excel("Base_ER_FR.xlsx")print(f"Dataset charge : {df.shape[0]} lignes, {df.shape[1]} colonnes")# Nettoyagecols_to_drop = ["Unnamed: 19", "MREVAU", "MREVNU"]df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)df["DARRET"] = pd.to_datetime(df["DARRET"].astype(str) + "01", format="%Y%m%d", errors="coerce")if "E_EAD" in df.columns:    df["EAD"] = df["E_EAD"]    df.drop(columns=["E_EAD", "E_ONB"], inplace=True, errors="ignore")df = df[df.get("EAD", 1) > 0]df["RA"] = df["RA"].fillna(0)for c in ["CLASSACT", "CSP", "produit"]:    if c in df.columns:        df[c] = df[c].astype("category")df["Tx"] = df.groupby("produit", observed=True)["Tx"].transform(lambda x: x.fillna(x.median()))df["Tx"] = df["Tx"].fillna(df["Tx"].median())df["CSP"] = df["CSP"].cat.add_categories("Inconnu")df["CSP"] = df["CSP"].fillna("Inconnu")df["AGE_CLI"] = df["AGE_CLI"].fillna(df["AGE_CLI"].median())df["MREVTOT"] = df["MREVTOT"].fillna(df["MREVTOT"].median())df["AGE_PRET"] = df["B_MAT"] - df["B_RESMAT"]df = df[(df["AGE_PRET"] >= 0) & (df["AGE_PRET"] <= df["B_MAT"])]df["HORIZON_RES"] = df["B_RESMAT"].astype(int)print("Preparation terminee")

## Etape 2 : Calcul de l'encours et de l'ER

### Formule de l'encours

L'encours est calcule comme la **valeur actuelle des mensualites futures** :

$$Encours = MTECH \times \frac{1-(1+r)^{-B\_RESMAT}}{r}$$

ou $r = Tx/1200$ est le taux mensuel.

**Source** : Vernimmen (2023), Brealey-Myers (2020), IFRS 9 paragraphe 5.4.1

In [ ]:
# Calcul encours (methode VP annuites - IFRS 9)df["Taux_mensuel"] = df["Tx"] / 1200df["Encours"] = df["MTECH"] * (    (1 - (1 + df["Taux_mensuel"]) ** (-df["B_RESMAT"])) / df["Taux_mensuel"])# Cas particulier : B_RESMAT = 0df.loc[df["B_RESMAT"] == 0, "Encours"] = 0# Calcul ERdf["ER_obs"] = (df["RA"] / df["Encours"]).clip(lower=0, upper=1)df["FLAG_ER"] = (df["RA"] > 0).astype(int)# Filtragedf = df[(df["Encours"] > 0)]print(f"Donnees preparees : {len(df)} contrats")print(f"ER_obs moyen : {df['ER_obs'].mean():.4f}")print(f"FLAG_ER taux : {df['FLAG_ER'].mean():.2%}")print(f"\nStatistiques ER_obs :")print(df['ER_obs'].describe())

## Etape 3 : Definition des etats de la chaine de Markov

### Choix des etats

On discretise l'ER en 4 etats :

| Etat | Label | Condition | Interpretation |
|------|-------|-----------|----------------|
| 1 | Pas de RA | ER = 0 | Aucun remboursement anticipe |
| 2 | RA faible | 0 < ER <= 0.33 | Remboursement partiel faible |
| 3 | RA moyen | 0.33 < ER <= 0.67 | Remboursement partiel significatif |
| 4 | RA fort | ER > 0.67 | Remboursement important ou total |

### Choix des tranches d'age

On divise l'age du pret en tranches pour estimer des matrices de transition specifiques :
- Tranche 1 : 0-12 mois (debut de vie du pret)
- Tranche 2 : 13-24 mois
- Tranche 3 : 25-36 mois
- Tranche 4 : 37-60 mois
- Tranche 5 : > 60 mois (prets anciens)

In [ ]:
# Definition des etatsdef assign_state(er):    if er == 0:        return 1  # Pas de RA    elif er <= 0.33:        return 2  # RA faible    elif er <= 0.67:        return 3  # RA moyen    else:        return 4  # RA fortdf['Etat'] = df['ER_obs'].apply(assign_state)# Labels des etatsstate_labels = {    1: 'Pas de RA (ER=0)',    2: 'RA faible (0<ER<=0.33)',    3: 'RA moyen (0.33<ER<=0.67)',    4: 'RA fort (ER>0.67)'}# Definition des tranches d'agebins = [0, 12, 24, 36, 60, float('inf')]labels_age = ['0-12m', '13-24m', '25-36m', '37-60m', '>60m']df['Tranche_age'] = pd.cut(df['AGE_PRET'], bins=bins, labels=labels_age, right=True)print("DISTRIBUTION DES ETATS")print("=" * 60)for etat, label in state_labels.items():    n = (df['Etat'] == etat).sum()    pct = (df['Etat'] == etat).mean()    print(f"  Etat {etat} ({label}) : {n} contrats ({pct:.1%})")print("\nDISTRIBUTION PAR TRANCHE D'AGE")print("=" * 60)print(df.groupby('Tranche_age', observed=True)['Etat'].value_counts(normalize=True).unstack().fillna(0).round(3))

### Visualisation distribution des etats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))# Distribution globale des etatsetat_counts = df['Etat'].value_counts().sort_index()colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']axes[0].bar([state_labels[e] for e in etat_counts.index],             etat_counts.values, color=colors, edgecolor='black')axes[0].set_xlabel('Etat', fontsize=11)axes[0].set_ylabel('Nombre de contrats', fontsize=11)axes[0].set_title('Distribution globale des etats de RA', fontsize=13)axes[0].tick_params(axis='x', rotation=30)axes[0].grid(True, alpha=0.3, axis='y')for i, (etat, count) in enumerate(zip(etat_counts.index, etat_counts.values)):    axes[0].text(i, count + 50, f'{count/len(df):.1%}', ha='center', fontsize=10)# Distribution des etats par tranche d'agedist_age_etat = df.groupby('Tranche_age', observed=True)['Etat'].value_counts(normalize=True).unstack().fillna(0)dist_age_etat.plot(kind='bar', ax=axes[1], color=colors, edgecolor='black')axes[1].set_xlabel('Tranche d'age', fontsize=11)axes[1].set_ylabel('Proportion', fontsize=11)axes[1].set_title('Distribution des etats par tranche d'age', fontsize=13)axes[1].legend([state_labels[e] for e in [1,2,3,4]],                loc='upper right', fontsize=8, title='Etat')axes[1].tick_params(axis='x', rotation=30)axes[1].grid(True, alpha=0.3, axis='y')plt.tight_layout()plt.show()print("Interpretation :")print("  - L'etat dominant change selon l'age du pret")print("  - Les RA forts sont plus frequents a certains ages")print("  - La distribution evolue au fil du temps")

## Etape 4 : Estimation des matrices de transition

### Methode d'estimation

Pour chaque tranche d'age, on estime la matrice de transition $M(t)$ comme :

$$p_{ij}(t) = \frac{n_{ij}(t)}{n_i(t)}$$

ou :
- $n_{ij}(t)$ = nombre de contrats passant de l'etat i a l'etat j dans la tranche t
- $n_i(t)$ = nombre total de contrats dans l'etat i dans la tranche t

### Contrainte de triangularite

La matrice est **triangulaire superieure** : on ne peut qu'augmenter son niveau de RA (les prêts ne "reviennent pas en arriere").

Dans notre cas avec une cross-section, on estime chaque matrice **independamment** par tranche d'age.

In [ ]:
def estimate_transition_matrix(df_slice, n_states=4):    """    Estime la matrice de transition pour une tranche d'age donnee.        Dans une cross-section, on estime la distribution des etats    et on construit une matrice triangulaire superieure basee sur    les probabilites conditionnelles observees.        Parameters:    -----------    df_slice : DataFrame pour une tranche d'age    n_states : nombre d'etats        Returns:    --------    numpy array (n_states x n_states)    """    # Distribution des etats dans cette tranche    state_dist = df_slice['Etat'].value_counts(normalize=True).reindex(        range(1, n_states + 1), fill_value=0    )        # ER moyen par etat dans cette tranche    er_by_state = df_slice.groupby('Etat')['ER_obs'].mean().reindex(        range(1, n_states + 1), fill_value=0    )        # Construire matrice triangulaire superieure    # P(rester dans etat i) = proportion dans etat i    # P(monter vers etat j>i) = distribuee proportionnellement    M = np.zeros((n_states, n_states))        for i in range(n_states):        etat_i = i + 1                # Proportion dans cet etat        p_i = state_dist[etat_i]                if p_i > 0:            # Probabilite de rester dans etat i            M[i, i] = p_i                        # Probabilite de monter vers etats superieurs            p_sup = 1 - p_i            states_sup = range(i + 1, n_states)                        if len(list(states_sup)) > 0:                for j in states_sup:                    etat_j = j + 1                    p_j = state_dist[etat_j]                    if p_j > 0:                        M[i, j] = p_sup * (p_j / state_dist[range(etat_i + 1, n_states + 1)].sum()) if state_dist[range(etat_i + 1, n_states + 1)].sum() > 0 else 0        else:            M[i, i] = 1.0        # Normaliser chaque ligne pour sommer a 1    for i in range(n_states):        row_sum = M[i, :].sum()        if row_sum > 0:            M[i, :] = M[i, :] / row_sum        return M, state_dist, er_by_state# Estimer les matrices par tranche d'agematrices = {}distributions = {}er_by_state_by_age = {}print("MATRICES DE TRANSITION PAR TRANCHE D'AGE")print("=" * 80)for tranche in labels_age:    df_slice = df[df['Tranche_age'] == tranche]        if len(df_slice) >= 30:        M, dist, er_state = estimate_transition_matrix(df_slice)        matrices[tranche] = M        distributions[tranche] = dist        er_by_state_by_age[tranche] = er_state                print(f"\nTranche {tranche} ({len(df_slice)} contrats) :")        print("Distribution des etats :")        for etat in range(1, 5):            print(f"  Etat {etat} : {dist[etat]:.3f} ({dist[etat]:.1%})")        print(f"\nMatrice de transition :")        df_matrix = pd.DataFrame(M,                                   index=[f'Etat {i+1}' for i in range(4)],                                  columns=[f'Etat {i+1}' for i in range(4)])        print(df_matrix.round(4))    else:        print(f"\nTranche {tranche} : pas assez de donnees ({len(df_slice)} contrats)")

### Visualisation des matrices de transition

In [ ]:
fig, axes = plt.subplots(1, len(matrices), figsize=(5 * len(matrices), 5))if len(matrices) == 1:    axes = [axes]state_labels_short = ['Etat 1\n(Pas RA)', 'Etat 2\n(Faible)', 'Etat 3\n(Moyen)', 'Etat 4\n(Fort)']for idx, (tranche, M) in enumerate(matrices.items()):    ax = axes[idx]        mask = np.tril(np.ones_like(M, dtype=bool), k=-1)        sns.heatmap(M, annot=True, fmt='.3f', cmap='Blues',                xticklabels=state_labels_short,                yticklabels=state_labels_short,                ax=ax, vmin=0, vmax=1,                linewidths=0.5, linecolor='gray',                mask=mask)        ax.set_title(f'Matrice de transition\nTranche {tranche}', fontsize=11)    ax.set_xlabel('Etat suivant', fontsize=9)    ax.set_ylabel('Etat actuel', fontsize=9)plt.suptitle('Matrices de transition par tranche d\'age\n(Structure triangulaire superieure)',              fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.show()print("Interpretation :")print("  - Diagonale : probabilite de rester dans le meme etat")print("  - Triangle superieur : probabilite de passer a un etat superieur")print("  - Triangle inferieur masque : pas de retour en arriere (absente)")print("  - Chaque ligne somme a 1")

## Etape 5 : Matrice de transition globale

On estime une matrice de transition **globale** sur l'ensemble des donnees, puis on analyse son evolution par tranche d'age.

In [ ]:
# Matrice de transition globale (tous ages confondus)M_global, dist_global, er_global = estimate_transition_matrix(df)print("MATRICE DE TRANSITION GLOBALE")print("=" * 80)print("\nDistribution globale des etats :")for etat in range(1, 5):    label = state_labels[etat]    print(f"  Etat {etat} ({label}) : {dist_global[etat]:.3f} ({dist_global[etat]:.1%})")print("\nMatrice de transition globale M :")df_M_global = pd.DataFrame(M_global,                            index=[f'De Etat {i+1}' for i in range(4)],                            columns=[f'Vers Etat {i+1}' for i in range(4)])print(df_M_global.round(4))# Verifier propriete stochastique (somme lignes = 1)print("\nVerification : somme des lignes (doit etre = 1) :")for i in range(4):    print(f"  Ligne {i+1} : {M_global[i,:].sum():.4f}")# Visualisation matrice globalefig, ax = plt.subplots(figsize=(8, 6))sns.heatmap(M_global, annot=True, fmt='.3f', cmap='Blues',            xticklabels=[state_labels_short[i] for i in range(4)],            yticklabels=[state_labels_short[i] for i in range(4)],            ax=ax, vmin=0, vmax=1,            linewidths=0.5, linecolor='gray',            annot_kws={"size": 12})ax.set_title('Matrice de transition GLOBALE\n(Tous ages confondus)', fontsize=14)ax.set_xlabel('Etat suivant', fontsize=12)ax.set_ylabel('Etat actuel', fontsize=12)plt.tight_layout()plt.show()

## Etape 6 : Analyse de l'evolution des probabilites de transition

On analyse comment les probabilites de transition evoluent avec l'age du pret.

Cette analyse permet de repondre a la question :
**Est-ce que le risque de RA change selon l'age du pret ?**

In [ ]:
# Evolution des probabilites de transition selon l'agefig, axes = plt.subplots(2, 2, figsize=(16, 12))# Pour chaque etat de departfor i_etat in range(4):    ax = axes[i_etat // 2, i_etat % 2]        tranches = []    probs_by_dest = {j: [] for j in range(4)}        for tranche, M in matrices.items():        tranches.append(tranche)        for j in range(4):            probs_by_dest[j].append(M[i_etat, j])        colors_dest = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']        for j in range(4):        ax.plot(tranches, probs_by_dest[j], 'o-',                 linewidth=2, markersize=8,                color=colors_dest[j],                label=f'Vers Etat {j+1}')        ax.set_xlabel('Tranche d\'age', fontsize=11)    ax.set_ylabel('Probabilite de transition', fontsize=11)    ax.set_title(f'Transitions depuis Etat {i_etat+1}\n({state_labels[i_etat+1]})', fontsize=12)    ax.legend(fontsize=9)    ax.set_ylim(0, 1)    ax.grid(True, alpha=0.3)plt.suptitle('Evolution des probabilites de transition par age du pret',              fontsize=14, fontweight='bold')plt.tight_layout()plt.show()print("Interpretation :")print("  - Si les courbes sont stables : les transitions ne dependent pas de l'age")print("  - Si les courbes varient : l'age du pret influence le comportement de RA")

## Etape 7 : Projection de l'ER par age

### Principe de la projection

A partir d'une distribution initiale $\pi(0)$, on projette l'ER attendu a chaque age :

$$E[ER(t)] = \sum_{s=1}^{4} \pi_s(t) \times \bar{ER}_s$$

ou $\bar{ER}_s$ est l'ER moyen observe dans l'etat s.

In [ ]:
# ER moyen par etat (global)er_mean_by_state = df.groupby('Etat')['ER_obs'].mean().reindex(range(1, 5), fill_value=0)print("ER MOYEN PAR ETAT")print("=" * 60)for etat in range(1, 5):    print(f"  Etat {etat} ({state_labels[etat]}) : ER moyen = {er_mean_by_state[etat]:.4f}")# Distribution initiale : proportion dans chaque etat au debut (age = 0)# On prend la distribution observee pour les jeunes prets (0-12 mois)if '0-12m' in distributions:    pi_0 = np.array([distributions['0-12m'][i] for i in range(1, 5)])else:    pi_0 = np.array([dist_global[i] for i in range(1, 5)])print(f"\nDistribution initiale pi(0) :")for i, p in enumerate(pi_0):    print(f"  Etat {i+1} : {p:.4f} ({p:.1%})")# Projection sur 60 moisn_mois = 60er_projected = []pi_t = pi_0.copy()# Utiliser matrice globale pour la projectionM_proj = M_global.copy()for t in range(n_mois):    # ER attendu a cet age    er_t = sum(pi_t[s] * er_mean_by_state[s+1] for s in range(4))    er_projected.append(er_t)        # Mise a jour distribution    pi_t = pi_t @ M_projer_projected = np.array(er_projected)print(f"\nProjection ER sur {n_mois} mois :")print(f"  ER projete moyen : {er_projected.mean():.4f}")print(f"  ER projete a 12m : {er_projected[11]:.4f}")print(f"  ER projete a 24m : {er_projected[23]:.4f}")print(f"  ER projete a 36m : {er_projected[35]:.4f}")

### Visualisation de la projection

In [ ]:
# ER observe par age (pour comparaison)er_obs_by_age = df.groupby('AGE_PRET')['ER_obs'].agg(['mean', 'std', 'count']).reset_index()er_obs_by_age = er_obs_by_age[er_obs_by_age['count'] >= 5]fig, axes = plt.subplots(2, 1, figsize=(14, 12))# Graphique 1 : ER observe vs ER projeteages_proj = range(n_mois)axes[0].plot(ages_proj, er_projected, linewidth=3, color='steelblue',              label='ER projete (Markov)', zorder=5)axes[0].scatter(er_obs_by_age['AGE_PRET'], er_obs_by_age['mean'],                 alpha=0.5, s=30, color='gray', label='ER observe (moyenne)', zorder=3)axes[0].axhline(y=df['ER_obs'].mean(), color='red', linestyle='--', linewidth=2,                label=f'ER global moyen = {df["ER_obs"].mean():.3f}')axes[0].set_xlabel('Age du pret (mois)', fontsize=12)axes[0].set_ylabel('ER', fontsize=12)axes[0].set_title('ER projete par modele Markovien vs ER observe', fontsize=13)axes[0].legend(fontsize=11)axes[0].grid(True, alpha=0.3)axes[0].set_ylim(0, 1)# Graphique 2 : Evolution de la distribution des etats dans le tempspi_history = [pi_0.copy()]pi_t = pi_0.copy()for t in range(n_mois - 1):    pi_t = pi_t @ M_proj    pi_history.append(pi_t.copy())pi_history = np.array(pi_history)colors_states = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']for s in range(4):    axes[1].plot(range(n_mois), pi_history[:, s],                  linewidth=2.5, color=colors_states[s],                 label=state_labels[s+1])axes[1].set_xlabel('Age du pret (mois)', fontsize=12)axes[1].set_ylabel('Proportion dans chaque etat', fontsize=12)axes[1].set_title('Evolution de la distribution des etats selon l\'age', fontsize=13)axes[1].legend(fontsize=10)axes[1].grid(True, alpha=0.3)axes[1].set_ylim(0, 1)plt.tight_layout()plt.show()print("Interpretation :")print("  - La courbe bleue = ER projete par le modele Markovien")print("  - Les points gris = ER observe dans les donnees")print("  - Le graphique du bas montre comment la population evolue entre etats")

## Etape 8 : Matrices par segment (analyse approfondie)

On estime des matrices de transition **specifiques par segment** pour analyser les differences de comportement selon les caracteristiques des emprunteurs.

In [ ]:
# Segmentation par quartile de taux d'interetdf['Tx_quartile'] = pd.qcut(df['Tx'], q=4,                               labels=['Q1 (taux bas)', 'Q2', 'Q3', 'Q4 (taux haut)'],                              duplicates='drop')print("MATRICES DE TRANSITION PAR QUARTILE DE TAUX")print("=" * 80)matrices_par_segment = {}for quartile in df['Tx_quartile'].cat.categories:    df_q = df[df['Tx_quartile'] == quartile]    if len(df_q) >= 50:        M_q, dist_q, er_q = estimate_transition_matrix(df_q)        matrices_par_segment[quartile] = M_q                print(f"\nSegment {quartile} ({len(df_q)} contrats) :")        print(f"  Distribution : " + " | ".join([f"E{i+1}:{dist_q[i+1]:.2%}" for i in range(4)]))        print(f"  ER moyen par etat : " + " | ".join([f"E{i+1}:{er_q[i+1]:.3f}" for i in range(4)]))# Visualisation comparaison matrices par segmentfig, axes = plt.subplots(1, len(matrices_par_segment),                           figsize=(5 * len(matrices_par_segment), 5))if len(matrices_par_segment) == 1:    axes = [axes]for idx, (segment, M_seg) in enumerate(matrices_par_segment.items()):    ax = axes[idx]    sns.heatmap(M_seg, annot=True, fmt='.2f', cmap='YlOrRd',                xticklabels=[f'E{i+1}' for i in range(4)],                yticklabels=[f'E{i+1}' for i in range(4)],                ax=ax, vmin=0, vmax=1,                linewidths=0.5)    ax.set_title(f'Matrice - {segment}', fontsize=10)    ax.set_xlabel('Etat suivant', fontsize=9)    ax.set_ylabel('Etat actuel', fontsize=9)plt.suptitle('Matrices de transition par segment de taux d\'interet',              fontsize=13, fontweight='bold')plt.tight_layout()plt.show()print("\nInterpretation :")print("  - Comparer les matrices entre segments montre l'effet du taux")print("  - Taux eleve = plus de RA (refinancement avantageux)")print("  - Differences entre matrices = heterogeneite des comportements")

## Etape 9 : Prediction individuelle avec le modele Markovien

### Principe

Pour chaque contrat, on :
1. Identifie son etat actuel (selon ER_obs)
2. Applique la matrice de transition correspondant a sa tranche d'age
3. Predit l'ER attendu au prochain horizon

$$E[ER_{predit}] = \sum_{j=1}^{4} p_{ij}(t) \times \bar{ER}_j$$

ou $i$ est l'etat actuel du contrat.

In [ ]:
# Prediction individuelledef predict_er_markov(etat_actuel, tranche_age, matrices, er_mean_by_state, M_global):    """    Predit l'ER attendu pour un contrat selon son etat actuel et sa tranche d'age.        Parameters:    -----------    etat_actuel : int (1-4)    tranche_age : str    matrices : dict de matrices par tranche    er_mean_by_state : Series    M_global : matrice globale (fallback)        Returns:    --------    float : ER predit    """    # Selectionner la matrice appropriee    if tranche_age in matrices:        M = matrices[tranche_age]    else:        M = M_global        # Ligne correspondant a l'etat actuel    i = etat_actuel - 1    prob_row = M[i, :]        # ER attendu = somme pondéree    er_predit = sum(prob_row[j] * er_mean_by_state[j+1] for j in range(4))        return er_predit# Appliquer les predictionsdf['ER_predit_Markov'] = df.apply(    lambda row: predict_er_markov(        row['Etat'],         row['Tranche_age'],        matrices,        er_mean_by_state,        M_global    ),     axis=1)# Evaluationfrom sklearn.metrics import mean_squared_error, mean_absolute_error, r2_scorefrom sklearn.model_selection import train_test_split# Split train/testX_train, X_test, y_train, y_test = train_test_split(    df.index, df['ER_obs'], test_size=0.2, random_state=42)y_pred_markov = df.loc[X_test, 'ER_predit_Markov'].valuesmse_markov = mean_squared_error(y_test, y_pred_markov)mae_markov = mean_absolute_error(y_test, y_pred_markov)r2_markov = r2_score(y_test, y_pred_markov)print("PERFORMANCE - MODELE MARKOVIEN")print("=" * 60)print(f"RMSE : {np.sqrt(mse_markov):.4f}")print(f"MAE  : {mae_markov:.4f}")print(f"R2   : {r2_markov:.4f}")print()print("Interpretation :")if r2_markov < 0.15:    print(f"  R2 = {r2_markov:.3f} : Performance similaire a Chain Ladder")    print("  La matrice de transition capte les patterns agregés")    print("  Mais pas les caracteristiques individuelles")elif r2_markov < 0.40:    print(f"  R2 = {r2_markov:.3f} : Performance correcte")    print("  Le modele Markovien capte une part significative de la variance")else:    print(f"  R2 = {r2_markov:.3f} : Bonne performance")

### Visualisation predictions Markov

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))# Scatter plotaxes[0].scatter(y_test, y_pred_markov, alpha=0.3, s=20, edgecolors='black', linewidth=0.3)axes[0].plot([0, 1], [0, 1], 'r--', linewidth=2, label='Prediction parfaite')axes[0].set_xlabel('ER observe', fontsize=11)axes[0].set_ylabel('ER predit (Markov)', fontsize=11)axes[0].set_title(f'Modele Markovien - Predictions vs Realite\nR2 = {r2_markov:.3f}', fontsize=12)axes[0].set_xlim(0, 1)axes[0].set_ylim(0, 1)axes[0].legend()axes[0].grid(True, alpha=0.3)# Distribution ER predit vs observeaxes[1].hist(y_test, bins=50, alpha=0.5, label='ER observe', color='blue', edgecolor='black')axes[1].hist(y_pred_markov, bins=50, alpha=0.5, label='ER predit (Markov)', color='green', edgecolor='black')axes[1].set_xlabel('ER', fontsize=11)axes[1].set_ylabel('Frequence', fontsize=11)axes[1].set_title('Distribution ER observe vs predit', fontsize=12)axes[1].legend()axes[1].grid(True, alpha=0.3, axis='y')plt.tight_layout()plt.show()residuals = y_test.values - y_pred_markovprint(f"Moyenne residus : {residuals.mean():.4f}")print(f"Std residus     : {residuals.std():.4f}")

## Etape 10 : Analyse de sensibilite

On analyse comment l'ER projete change selon :
1. La distribution initiale des etats
2. Les parametres de la matrice

In [ ]:
# Sensibilite a la distribution initialefig, axes = plt.subplots(1, 2, figsize=(16, 6))# Scenario 1 : Tous dans Etat 1 (Pas de RA)pi_S1 = np.array([1.0, 0.0, 0.0, 0.0])# Scenario 2 : Distribution observeepi_S2 = pi_0.copy()# Scenario 3 : Distribution plus risqueepi_S3 = np.array([0.5, 0.2, 0.2, 0.1])scenarios = {    'S1 : Tous sans RA': pi_S1,    'S2 : Distribution observee': pi_S2,    'S3 : Distribution risquee': pi_S3}colors_scenarios = ['green', 'blue', 'red']for ax_idx, (titre, pi_init) in enumerate(scenarios.items()):    er_proj_scenario = []    pi_t_scenario = pi_init.copy()        for t in range(n_mois):        er_t = sum(pi_t_scenario[s] * er_mean_by_state[s+1] for s in range(4))        er_proj_scenario.append(er_t)        pi_t_scenario = pi_t_scenario @ M_proj        axes[0].plot(range(n_mois), er_proj_scenario,                  linewidth=2.5, label=titre)axes[0].set_xlabel('Age du pret (mois)', fontsize=11)axes[0].set_ylabel('ER projete', fontsize=11)axes[0].set_title('Sensibilite a la distribution initiale', fontsize=12)axes[0].legend(fontsize=9)axes[0].grid(True, alpha=0.3)axes[0].set_ylim(0, 1)# Etat stationnaire (long terme)# Calculer distribution stationnaire : pi × M = pifrom numpy.linalg import eig# Distribution stationnaire = vecteur propre gauche associe a valeur propre 1eigenvalues, eigenvectors = eig(M_global.T)idx_stationary = np.argmin(np.abs(eigenvalues - 1.0))pi_stationary = np.real(eigenvectors[:, idx_stationary])pi_stationary = pi_stationary / pi_stationary.sum()axes[1].bar([state_labels_short[i] for i in range(4)], pi_stationary,             color=colors_states, edgecolor='black')axes[1].set_xlabel('Etat', fontsize=11)axes[1].set_ylabel('Probabilite stationnaire', fontsize=11)axes[1].set_title('Distribution stationnaire (long terme)', fontsize=12)axes[1].grid(True, alpha=0.3, axis='y')for i, p in enumerate(pi_stationary):    axes[1].text(i, p + 0.01, f'{p:.2%}', ha='center', fontsize=10)plt.tight_layout()plt.show()print("DISTRIBUTION STATIONNAIRE (etat d'equilibre long terme) :")for i, p in enumerate(pi_stationary):    print(f"  Etat {i+1} ({state_labels[i+1]}) : {p:.4f} ({p:.1%})")print()print("Interpretation :")print("  La distribution stationnaire represente l'equilibre long terme")print("  Si pi(t) converge vers cette distribution, le systeme est stable")

## Synthese et limites du modele Markovien

### Points forts

1. **Approche matricielle rigoureuse**
   - Matrice triangulaire superieure = coherence economique
   - Interpretable et explicable aux decideurs

2. **Compatible avec donnees cross-section**
   - Pas besoin de suivi longitudinal
   - Une observation par contrat suffit

3. **Analyse de sensibilite naturelle**
   - Facilement simulable
   - Scenarios stress testing possibles

4. **Distribution stationnaire**
   - Donne l'equilibre long terme du portefeuille
   - Utile pour planification ALM

5. **Segmentation possible**
   - Matrices differentes par segment (taux, produit, CSP)
   - Capture l'heterogeneite du portefeuille

### Limites

1. **Propriete de Markov peut etre violee**
   - Le comportement de RA peut dependre de l'historique
   - Pas uniquement de l'etat actuel

2. **Discretisation des etats**
   - Le choix des seuils (0.33, 0.67) est arbitraire
   - Sensibilite aux seuils choisis

3. **Performance predictive limitee**
   - R2 typiquement faible (similaire a Chain Ladder)
   - Ne capte pas les caracteristiques individuelles

4. **Hypothese de stationnarite**
   - Les matrices sont supposees stables dans le temps
   - Peut ne pas etre vrai en periode de changement de taux

5. **Estimation sur cross-section**
   - Idealement : suivi longitudinal des memes contrats
   - Notre estimation est une approximation

### Recommandations

**Utiliser le modele Markovien pour** :
- Analyse exploratoire et descriptive
- Projections agregees du portefeuille
- Stress testing et scenarios ALM
- Communication avec les decideurs (matrice intuitive)

**Combiner avec ML pour** :
- Predictions individuelles precises
- Integration des caracteristiques individuelles
- Meilleure performance predictive

### Comparaison avec les autres approches

| Approche | Performance | Interpretabilite | Temporalite | Adapte donnees |
|----------|-------------|------------------|-------------|----------------|
| Chain Ladder | R2 ~ 0.09 | Elevee | Oui | Non (longitudinal) |
| Markov | R2 ~ 0.10-0.15 | Elevee | Oui | Oui (cross-section) |
| Cox | C-index ~ 0.75 | Elevee (HR) | Oui | Oui |
| ML pur | R2 ~ 0.60-0.70 | Moyenne | Non | Oui |

### Suite

**Notebook 2** : Survival Analysis (Cox, Kaplan-Meier)
**Notebook 3** : Machine Learning (Random Forest)
**Notebook 4** : Modele Hybride (Markov + ML)

## Sauvegarde des resultats

In [ ]:
# Sauvegarder matrices de transitionfor tranche, M in matrices.items():    df_M = pd.DataFrame(M,                        index=[f'Etat_{i+1}' for i in range(4)],                        columns=[f'Etat_{i+1}' for i in range(4)])    df_M.to_csv(f'matrice_transition_{tranche}.csv')# Sauvegarder matrice globaledf_M_global = pd.DataFrame(M_global,                            index=[f'Etat_{i+1}' for i in range(4)],                            columns=[f'Etat_{i+1}' for i in range(4)])df_M_global.to_csv('matrice_transition_globale.csv')# Sauvegarder projectionpd.DataFrame({    'Age_mois': range(n_mois),    'ER_projete': er_projected}).to_csv('markov_projection_er.csv', index=False)# Sauvegarder performancepd.DataFrame({    'Approche': ['Modele Markovien'],    'R2': [r2_markov],    'RMSE': [np.sqrt(mse_markov)],    'MAE': [mae_markov]}).to_csv('markov_performance.csv', index=False)print("Fichiers sauvegardes :")print("  - matrice_transition_globale.csv")for tranche in matrices.keys():    print(f"  - matrice_transition_{tranche}.csv")print("  - markov_projection_er.csv")print("  - markov_performance.csv")